# MLP Adam

### Importamos librerias necesarias

In [8]:
import numpy as np
import pandas as pd

### Concepto 1: Función de pérdida
`Binary Cross-Entropy` 

In [9]:
def binary_cross_entropy(y_true, y_pred, epsilon = 1e-8):
    """
    y_true: array (m,) o (m, 1) de etiquetas 0/1
    y_pred: array (m, 1) de probabilidades predichas
    """
    y_true = y_true.reshape(-1, 1)
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)  # evitar log(0)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

### Concepto 2: Funciones de activación

In [10]:
# Usada en el forward pass
def relu(Z):
    return np.maximum(0, Z)

# Necesaria para el backward pass
def relu_derivative(Z):
    # Derivada de ReLU: 1 si Z > 0, 0 en caso contrario
    return (Z > 0).astype(Z.dtype)


def sigmoid(Z):
    # Clipping para evitar overflow en np.exp con valores extremos
    Z = np.clip(Z, -500, 500)
    return 1.0 / (1.0 + np.exp(-Z))

### Concepto 3: Momento estadísticos
`Inicialización He:` Se usa la inicialización He ya que estamos usando ReLu como función de activación. 

In [11]:
def he_init(n_in, n_out, rng):
    """Pesos ~ Normal(0, sqrt(2/n_in)), forma (n_in, n_out)."""
    return rng.standard_normal((n_in, n_out)) * np.sqrt(2.0 / n_in)

### Concepto 4 y 5: Perceptron Multicapa (MLP) con Adam como optimizador

In [ ]:
class MLP:
    """
    MLP totalmente conectado para clasificacion binaria.

    layer_sizes : lista de enteros, p.ej. [n_features, 64, 32, 1]
    seed        : semilla para reproducibilidad de la inicializacion
    """

    def __init__(self, layer_sizes, seed = 42):
        self.layer_sizes = layer_sizes
        self.n_layers = len(layer_sizes) - 1
        rng = np.random.default_rng(seed)

        # Pesos y biases por capa
        self.W = []  # cada W[i] tiene forma (n_in, n_out)
        self.b = []  # cada b[i] tiene forma (1, n_out) para broadcasting
        for i in range(self.n_layers):
            n_in, n_out = layer_sizes[i], layer_sizes[i + 1]
            self.W.append(he_init(n_in, n_out, rng))
            self.b.append(np.zeros((1, n_out)))

        # Estado de Adam: primer y segundo momento por parametro
        self._init_adam_state()

    def _init_adam_state(self):
        self.mW = [np.zeros_like(W) for W in self.W]
        self.vW = [np.zeros_like(W) for W in self.W]
        self.mb = [np.zeros_like(b) for b in self.b]
        self.vb = [np.zeros_like(b) for b in self.b]
        self.t = 0  # contador de pasos de Adam (para bias correction)

    # ------------------------------------------------------------------
    # FORWARD PASS
    # ------------------------------------------------------------------
    def forward(self, X):
        """
        X: array (m, n_features).
        Cachea activaciones y pre-activaciones para el backward pass.
        """
        activations = [X]   # activacion en cada capa (incluyendo entrada)
        zs = []             # pre-activaciones (antes de la no-linealidad)

        current_activation = X
        for i in range(self.n_layers):
            # Usamos Z por nomenclatura usada en texto académicos y consistencia con la manera en la cual definimos las funciones de activación
            Z = current_activation @ self.W[i] + self.b[i]   # broadcasting suma el bias a cada fila
            zs.append(Z)

            if i < self.n_layers - 1:
                current_activation = relu(Z)         # capas ocultas
            else:
                current_activation = sigmoid(Z)      # capa de salida (clasificacion binaria)

            activations.append(current_activation)

        self._cache = {"activations": activations, "zs": zs}
        return current_activation

    # ------------------------------------------------------------------
    # BACKWARD PASS (gradientes manuales con regla de la cadena)
    # ------------------------------------------------------------------
    def backward(self, y_true):
        """
        y_true: array (m,) o (m, 1) de etiquetas 0/1.
        Devuelve listas de gradientes para W y b.
        """
        activations = self._cache["activations"]
        zs = self._cache["zs"]
        m = y_true.shape[0]
        y_true = y_true.reshape(-1, 1)

        grads_W = [None] * self.n_layers
        grads_b = [None] * self.n_layers

        # Truco: la derivada combinada de BCE + sigmoid se simplifica a
        # dL/dZ = (y_pred - y_true) / m. Numericamente estable y eficiente.
        y_pred = activations[-1]
        dZ = (y_pred - y_true) / m   # shape (m, 1)

        # Retropropagar capa por capa, de la ultima a la primera
        for layer in reversed(range(self.n_layers)):
            previous_activation = activations[layer]

            grads_W[layer] = previous_activation.T @ dZ                    # dL/dW
            grads_b[layer] = np.sum(dZ, axis = 0, keepdims = True)  # dL/db

            if layer > 0:
                dA_prev = dZ @ self.W[layer].T
                dZ = dA_prev * relu_derivative(zs[layer - 1])

        return grads_W, grads_b

    # ------------------------------------------------------------------
    # OPTIMIZADOR ADAM (implementado a mano, sin usar APIs de entrenamiento de Keras/Pytorch ni nada)
    # ------------------------------------------------------------------
    def adam_step(self, grads_W, grads_b,
                  lr = 0.001, beta1 = 0.9, beta2 = 0.999, epsilon = 1e-8):
        """
        Un paso de Adam:
            m_t = beta1 * m_{t-1} + (1-beta1) * g
            v_t = beta2 * v_{t-1} + (1-beta2) * g^2
            m_hat = m_t / (1 - beta1^t)        # bias correction
            v_hat = v_t / (1 - beta2^t)
            param -= lr * m_hat / (sqrt(v_hat) + epsilon)

        Con NumPy todas las operaciones son vectorizadas: no necesitamos
        loops sobre los elementos de las matrices.
        """
        self.t += 1
        bc1 = 1 - beta1 ** self.t
        bc2 = 1 - beta2 ** self.t

        for layer in range(self.n_layers):
            # --- Pesos ---
            gW = grads_W[layer]
            self.mW[layer] = beta1 * self.mW[layer] + (1 - beta1) * gW
            self.vW[layer] = beta2 * self.vW[layer] + (1 - beta2) * (gW ** 2)
            m_hat = self.mW[layer] / bc1
            v_hat = self.vW[layer] / bc2
            self.W[layer] -= lr * m_hat / (np.sqrt(v_hat) + epsilon)

            # --- Biases ---
            gb = grads_b[layer]
            self.mb[layer] = beta1 * self.mb[layer] + (1 - beta1) * gb
            self.vb[layer] = beta2 * self.vb[layer] + (1 - beta2) * (gb ** 2)
            m_hat_b = self.mb[layer] / bc1
            v_hat_b = self.vb[layer] / bc2
            self.b[layer] -= lr * m_hat_b / (np.sqrt(v_hat_b) + epsilon)

    # ------------------------------------------------------------------
    # ENTRENAMIENTO
    # ------------------------------------------------------------------
    def fit(self, X, y, epochs = 100, batch_size = 32, lr = 0.001,
            verbose = True, seed = 0):
        rng = np.random.default_rng(seed)
        n_samples = X.shape[0]
        history = []

        for epoch in range(1, epochs + 1):
            # Mezclar el dataset cada epoca
            perm = rng.permutation(n_samples)
            X_shuf, y_shuf = X[perm], y[perm]

            epoch_losses = []
            for start in range(0, n_samples, batch_size):
                end = start + batch_size
                X_batch = X_shuf[start:end]
                y_batch = y_shuf[start:end]

                y_pred = self.forward(X_batch)
                loss = binary_cross_entropy(y_batch, y_pred)
                epoch_losses.append(loss)

                grads_W, grads_b = self.backward(y_batch)
                self.adam_step(grads_W, grads_b, lr = lr)

            avg_loss = float(np.mean(epoch_losses))
            history.append(avg_loss)

            if verbose and (epoch == 1 or epoch % max(1, epochs // 10) == 0):
                acc = self.accuracy(X, y)
                print(f"Epoch {epoch:4d}/{epochs} - loss: {avg_loss:.4f} "
                      f"- train_acc: {acc:.4f}")

        return history

    # ------------------------------------------------------------------
    # INFERENCIA
    # ------------------------------------------------------------------
    def predict_proba(self, X):
        return self.forward(X)

    def predict(self, X, threshold = 0.5):
        return (self.forward(X).ravel() >= threshold).astype(int)

    def accuracy(self, X, y):
        return float(np.mean(self.predict(X) == y))

### Pruebas

In [13]:
# Carga del dataset (separador pipe)
df = pd.read_csv('new_dataset.csv', sep='|', parse_dates=['Submit'])
df.head()

,ConsumedEnergyRaw,CPUTimeRAW,ReqCPUS,ReqMem,ReqNodes,ResvCPURAW,Submit,TimelimitRaw,Partition,Priority,QOS,State
0,3800474.0,4147584,1,100Mc,1,345743,2024-06-25 14:03:31,1440,nukwa-wide,7046,normal,TIMEOUT
1,0.0,4147968,1,100Mc,1,172857,2024-06-25 14:03:34,1440,nukwa-wide,7046,normal,TIMEOUT
2,3774906.0,4147392,1,100Mc,1,432175,2024-06-25 14:03:37,1440,nukwa-wide,7046,normal,TIMEOUT
3,3782559.0,4147536,1,100Mc,1,259290,2024-06-25 14:03:47,1440,nukwa-wide,7046,normal,TIMEOUT
4,3723128.0,4088688,1,100Mc,1,518590,2024-06-25 14:03:56,1440,nukwa-wide,7046,normal,COMPLETED
